
# Lesson 6: State Management and Reducers

## 1. Learning Objectives

By the end of this lesson, we should understand:

- How LangGraph updates shared state.
- The difference between default state updates and reducer-based updates.
- What reducers are and why they are useful.
- How `Annotated` associates a reducer with a state field.
- How Python's `operator.add` combines list updates.

## 2. What is a Reducer?

A reducer is a function that determines how LangGraph combines an existing state value with a new update.

By default, when a node returns a new value for a state key, the previous value is replaced.

For example:

Existing state:
`{"results": ["Python"]}`

New node update:
`{"results": ["LangGraph"]}`

Without a reducer, the final value becomes:
`["LangGraph"]`

This means the previous information is lost.

A reducer changes this behavior by defining how the old and new values should be merged.

## 3. Understanding operator.add

Python provides `operator.add`, which performs addition.

Its behavior depends on the data type:

- Numbers: adds the values.
- Strings: concatenates the strings.
- Lists: concatenates the lists.

Example:

```python
existing = ["Python"]
new = ["LangGraph"]

existing + new
```

Output:

```python
["Python", "LangGraph"]
```

We use this behavior as a reducer in our implementation.

## 4. Annotated in LangGraph

Python's `Annotated` allows us to attach additional metadata to a type.

Syntax:

```python
Annotated[data_type, reducer_function]
```

Our implementation:

```python
results: Annotated[list, operator.add]
```

Explanation:

- `results`: state field name.
- `list`: expected data type.
- `Annotated`: attaches reducer metadata.
- `operator.add`: combines existing and new list values.

LangGraph uses this reducer whenever the `results` field receives an update.

## 5. Practical Implementation: Research Workflow

We created a simple research workflow containing two sequential nodes:

1. `python_research`: returns information about Python.
2. `langgraph_research`: returns information about LangGraph.

Both nodes update the same `results` field.

Because the field uses `operator.add` as its reducer, their results are accumulated instead of replaced.

### Workflow

START → Python Research → LangGraph Research → END

### Expected Output

```python
{
    "results": [
        "Python is widely used in AI development.",
        "LangGraph helps build stateful AI workflows."
    ]
}
```

## 6. What We Learned

| Concept | Purpose |
|---|---|
| State | Stores shared workflow information |
| Default update | Replaces the previous value |
| Reducer | Defines how updates are combined |
| Annotated | Attaches reducer metadata to a state field |
| operator.add | Concatenates lists in this implementation |

## 7. Key Takeaway

Without a reducer, a new update normally replaces the previous value for that state key.

With a reducer, LangGraph follows the merge behavior defined by that reducer.

Reducers are particularly important in workflows where multiple nodes contribute information to shared state.

## 8. Next Lesson

Next: Parallel Execution in LangGraph.

We will explore how multiple nodes can execute in parallel and contribute updates to shared state.


In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
import operator

In [2]:
class ResearchState(TypedDict):
    results: Annotated[list, operator.add]

In [3]:
def python_research(state: ResearchState):
    return {
        "results": ["Python is widely used in AI development."]
    }


def langgraph_research(state: ResearchState):
    return {
        "results": ["LangGraph helps build stateful AI workflows."]
    }

In [4]:
builder = StateGraph(ResearchState)

builder.add_node("python_research", python_research)
builder.add_node("langgraph_research", langgraph_research)

builder.add_edge(START, "python_research")
builder.add_edge("python_research", "langgraph_research")
builder.add_edge("langgraph_research", END)

graph = builder.compile()

In [5]:
initial_state = {
    "results": []
}

final_state = graph.invoke(initial_state)

print(final_state)

{'results': ['Python is widely used in AI development.', 'LangGraph helps build stateful AI workflows.']}
